# 실습 3 — 데이터 분할

**결과를 믿을 수 있는가**

- **분할** — 모델 성능을 재기 위해 데이터를 나누는 일
- 잘 나눈 성능 → 현실을 예고 · 잘못 나눈 성능 → 현실을 가림
- **전처리도 모델 학습도 아님** — 결과를 믿을 수 있게 만드는 설계

### 이 노트북에서 확인할 것

| | 내용 | 담당 |
|---|---|---|
| 1 | 분할의 목적 — 학습용 · 검증용 · 평가용 | — |
| 2 | **랜덤 분할 vs 시간순 분할** — 실측 비교 | **사람** |
| 3 | 분할의 편향 — 층화 추출 | 자동(분류) |
| 4 | 한 번의 분할 → K번의 분할 (교차검증) | 자동 |
| 5 | 최종 분할과 점검 | **사람** |

---

- 회색 배경 — 실행 가능한 **코드 셀** · 흰 배경 — 설명
- 코드 셀 선택 → **Shift + Enter** → 실행 · 상단부터 순서대로
- ⏱ 표시 셀 — 학습이 돌아 시간 소요

---
## 0. 환경 준비 · 전처리 재현

- 실습 2의 전처리 결과를 그대로 사용
- 습도 0 → NaN · 운영 중단 행 제거 · 도메인 파생 변수

In [ ]:
import io, logging, warnings, contextlib
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")
logging.getLogger("autogluon").setLevel(logging.CRITICAL)
pd.set_option("display.max_columns", 50)

CSV     = "../SeoulBikeData.csv"     # 경로가 다르면 여기만 수정
LABEL   = "Rented Bike Count"
N_CPUS  = 16                       # 환경에 맞게 조정
LIMIT   = 180                      # 학습 시간 제한(초)
EXCLUDE = ["FASTAI"]              # 환경 오류 모델 제외

print("준비 완료")

In [ ]:
def add_fe(x):
    """도메인 파생 변수 — 사람이 만든다 (실습 2)"""
    x = x.copy()
    T, V, H = x["Temperature(°C)"], x["Wind speed (m/s)"], x["Humidity(%)"]
    wc = 13.12 + 0.6215*T - 11.37*(V*3.6)**0.16 + 0.3965*T*(V*3.6)**0.16
    x["체감온도"]   = np.where(T <= 10, wc, T)
    x["불쾌지수"]   = 0.81*T + 0.01*H*(0.99*T - 14.3) + 46.3
    dow = x["Date"].dt.dayofweek
    x["평일출퇴근"] = ((dow < 5) & x["Hour"].isin([7,8,9,17,18,19])).astype(int)
    x["비옴"]      = (x["Rainfall(mm)"] > 0).astype(int)
    return x


def preprocess(path=CSV, use_fe=True):
    """실습 2의 최종 전처리"""
    x = pd.read_csv(path, encoding="latin-1")
    x["Date"] = pd.to_datetime(x["Date"], format="%d/%m/%Y")   # ① 날짜 인식
    x.loc[x["Humidity(%)"] == 0, "Humidity(%)"] = np.nan       # ② 위장 결측
    x = (x[x["Functioning Day"] == "Yes"]                      # ③ 누수 행 제거
           .drop(columns=["Functioning Day"])
           .reset_index(drop=True))
    if use_fe:
        x = add_fe(x)                                          # ④ 도메인 파생
    return x


clean = preprocess()
print("전처리 후:", clean.shape)
clean.head(3)

---
## 1. 분할의 목적

### 왜 나누는가

- 학습에 쓴 데이터로 측정 → 이미 접한 사례를 다시 평가 → **실제보다 높게 측정**
- 학습에 쓰지 않은 데이터로 평가해야 **일반화 성능** 확인 가능

### 세 갈래

| | 역할 | 누가 나누는가 |
|---|---|---|
| **학습용** train | 모델 학습 | 연구자 |
| **검증용** valid | **모델 선택** — 어느 모델이 좋은지 · 앙상블 가중치 | **AutoGluon 자동** |
| **평가용** test | **최종 보고** | 연구자 |

### 실제 비율

- 연구자 — 학습용 **80%** / 평가용 **20%**
- AutoGluon — 학습용을 다시 쪼갬 · `holdout_frac=0.1` → **검증용은 학습용의 10%**
- → 전체 기준 **72 / 8 / 20**

| 따릉이 8,465행 | 학습 6,091 · 검증 677 · 평가 1,697 |
|---|---|

- → 연구자가 나누는 것은 **학습용 / 평가용 둘**

### 비율 정하기

- 일반적 비율 — 80/20 또는 70/30
- 학습용↑ = 더 잘 배움 · 평가용↑ = 측정이 안정적
- **기준은 비율보다 평가용의 절대 개수**

> ⚠️ **평가용을 보고 무언가를 바꾸는 순간 — 그 숫자는 성능이 아니게 됨**
> 모델도, 설정도, 변수도 — 되돌아가 고치면 평가용에 맞춘 것

---
## 2. 랜덤 분할 vs 시간순 분할 ⭐

**값의 문제 아님 · 나누는 방법의 문제**

- 따릉이 = 시간 순서가 있는 데이터
- 랜덤으로 나누면 — 6월 3일 14시·16시로 학습하고 **15시를 평가**
- 예측이 아니라 **사이 채우기**

In [ ]:
# 두 가지 방식으로 분할
d = clean.sort_values(["Date", "Hour"]).reset_index(drop=True)

# ① 랜덤 분할
tr_rand = d.sample(frac=0.8, random_state=0)
te_rand = d.drop(tr_rand.index)

# ② 시간순 분할 — 날짜 경계에서 자름
days = np.sort(d["Date"].unique())
cut  = days[int(len(days) * 0.8)]              # 80% 지점의 날짜
tr_time, te_time = d[d["Date"] < cut], d[d["Date"] >= cut]

print("랜덤   — 학습 %s / 평가 %s" % (tr_rand.shape, te_rand.shape))
print("시간순 — 학습 %s / 평가 %s" % (tr_time.shape, te_time.shape))
print()
print("랜덤   겹치는 날짜:", len(set(tr_rand["Date"]) & set(te_rand["Date"])), "일  ← 문제")
print("시간순 겹치는 날짜:", len(set(tr_time["Date"]) & set(te_time["Date"])), "일")

> ⚠️ **행 개수가 아니라 날짜 경계에서 자를 것**
> `iloc[:n]` 으로 자르면 경계일 하루가 양쪽에 걸침
> 같은 날 오전은 학습 · 오후는 평가 → **누수**

### ⏱ 실측 — 같은 데이터, 같은 모델, 나누는 방법만 다르게

- 학습 2회 · 약 6분 소요

In [ ]:
from sklearn.metrics import mean_absolute_error


def quiet_fit(train, test, tag):
    """학습 로그를 삼키고 R² · MAE 반환"""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        p = TabularPredictor(label=LABEL, eval_metric="r2",
                             path=f"ag_{tag}", verbosity=0) \
            .fit(train, time_limit=LIMIT,
                 ag_args_fit={"num_cpus": N_CPUS},
                 excluded_model_types=EXCLUDE)
        r2  = p.evaluate(test, silent=True)["r2"]
        mae = mean_absolute_error(test[LABEL], p.predict(test))
    return r2, mae


r2_rand, mae_rand = quiet_fit(tr_rand, te_rand, "split_rand")
r2_time, mae_time = quiet_fit(tr_time, te_time, "split_time")

avg_rand = te_rand[LABEL].mean()
avg_time = te_time[LABEL].mean()

print("=" * 60)
print("%-12s %8s %10s %12s" % ("분할 방식", "R²", "평균 오차", "오차/평균"))
print("-" * 60)
print("%-12s %8.3f %8.0f대 %11.1f%%" % ("랜덤 분할",   r2_rand, mae_rand, mae_rand/avg_rand*100))
print("%-12s %8.3f %8.0f대 %11.1f%%" % ("시간순 분할", r2_time, mae_time, mae_time/avg_time*100))
print("=" * 60)
print("오차 %.1f배  ·  규모 보정 후 %.1f배" %
      (mae_time/mae_rand, (mae_time/avg_time)/(mae_rand/avg_rand)))

In [ ]:
# 두 평가셋이 같은 집단인가
print("%-10s %8s %10s %10s" % ("", "개수", "평균", "표준편차"))
for name, te in [("랜덤", te_rand), ("시간순", te_time)]:
    t = te[LABEL]
    print("%-10s %8d %10.1f %10.1f" % (name, len(t), t.mean(), t.std()))

print()
print("계절 구성")
print("  랜덤   :", te_rand["Seasons"].value_counts().to_dict())
print("  시간순 :", te_time["Seasons"].value_counts().to_dict())

#### ⚠️ 두 평가셋은 같은 집단이 아님

- 랜덤 — 네 계절이 고르게 섞임
- 시간순 — **가을만** · 평균 대여량도 다름
- R²는 그 평가셋 자신의 분산이 기준 → 잣대가 다름

> **격차에는 두 가지가 섞여 있음**
> ① 누수 제거 효과 · ② 학습에 없던 계절을 예측하는 어려움
> → 규모 보정 후에도 약 2배 — **결론은 유지**

> **실무에서 마주하는 것이 정확히 이 상황** — 미래는 늘 학습에 없던 시기

### 내 데이터에서는 — 묶임의 단위

| 묶임의 단위 | 랜덤 분할하면 |
|---|---|
| **시간** — 반복 관측 | 미래로 과거를 예측 |
| **개인** — 같은 학생·환자를 여러 번 | 이미 본 사람을 다시 맞힘 |
| **집단** — 같은 학교·병원·실험실 | 그 집단의 특성을 외움 |

> **묶여 있는 단위로 나눠야 함** — 성능이 좋게 나오는데 새 대상에게는 안 통함
> 참고 — 개인·집단 단위 분할은 `sklearn.model_selection.GroupKFold`

---
## 3. 분할의 편향 — 층화 추출

### 문제 상황 — 분류 문제

- 예: 제품 200개 중 불량품 20개
- 무작위 분할 → 우연히 불량품이 한쪽에 집중 가능
- 평가용에 불량품 2~3개만 → 평가 결과 불안정
- 학습용에 불량품 부족 → 불량 패턴을 학습하지 못함

### 해결 — 층화 추출 (Stratified Sampling)

- 주요 그룹의 비율이 학습용·평가용에서 동일하도록 분할

In [ ]:
from sklearn.model_selection import train_test_split

demo = pd.DataFrame({"id": range(200), "불량": [1]*20 + [0]*180})
print("전체 불량률: %.1f%%" % (demo["불량"].mean()*100))
print()

for seed in [0, 1, 2]:
    _, te = train_test_split(demo, test_size=0.2, random_state=seed)
    print("무작위 seed=%d  평가용 불량 %2d개 (%.1f%%)" %
          (seed, te["불량"].sum(), te["불량"].mean()*100))
print()
for seed in [0, 1, 2]:
    _, te = train_test_split(demo, test_size=0.2, random_state=seed,
                             stratify=demo["불량"])
    print("층화   seed=%d  평가용 불량 %2d개 (%.1f%%)" %
          (seed, te["불량"].sum(), te["불량"].mean()*100))

#### 결과 해석

- 무작위 — seed에 따라 평가용 불량 개수가 흔들림
- 층화 — 항상 4개(10%) 유지

| 대규모 | 무작위 분할로도 양쪽 분포가 유사해짐 |
|---|---|
| **소규모** | 우연에 의한 쏠림 위험이 커짐 |

> AutoGluon — 내부 검증 분할에 층화 추출 자동 적용 (분류 문제)
> 연구자가 수행하는 평가용 분할은 직접 확인 필요

> ⚠️ **층화 추출은 비율을 맞춤 · 순서나 소속은 맞추지 못함**
> 시간·개인·집단으로 묶인 데이터는 **2장의 방법**으로
> 따릉이는 회귀 문제 → 층화 대상 아님

---
## 4. 한 번의 분할 → K번의 분할

**교차검증 (Cross Validation)**

### 단일 분할의 한계 — 측정값이 하나뿐

- 300행 → 학습 240 · 평가 60
- 얻는 것은 **측정값 1개**
- 그 값은 **어떤 60개가 평가용이 되었는가**에 의존
- → 우연히 높은지 낮은지 알 방법 없음
- → 두 모델을 비교해도 차이가 실력인지 우연인지 **구분 불가**

In [ ]:
# 분할을 바꿔가며 같은 데이터로 반복 측정
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

small = d.sample(n=300, random_state=0)          # 300행만 사용
X = small.drop(columns=[LABEL, "Date"]).copy()
y = small[LABEL]

# 문자열 열 → 정수 코드 (sklearn은 문자열 입력 불가)
for col in ["Seasons", "Holiday"]:
    X[col] = pd.Categorical(X[col], categories=sorted(X[col].dropna().unique())).codes

scores = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = HistGradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
    scores.append(r2_score(yte, m.predict(Xte)))

print("같은 데이터 · 같은 모델 · 분할 seed만 변경")
print()
for i, s in enumerate(scores):
    print("  seed=%d   R² = %.3f" % (i, s))
print()
print("최저 %.3f  ~  최고 %.3f   (폭 %.3f)" % (min(scores), max(scores), max(scores)-min(scores)))
print("평균 %.3f  ± %.3f" % (np.mean(scores), np.std(scores)))

#### 결과 해석

- **모델이 달라진 것이 아니라 평가용이 달라진 것**
- 측정값 1개로는 이 폭을 알 수 없음
- 두 모델 비교 시 — **분할 운으로 순위가 바뀔 수 있음**

### 해결 — 여러 번 나누고 평균

```
전체를 K조각으로 나눔  (예: K=5)

  1회 ■□□□□   2회 □■□□□   3회 □□■□□   4회 □□□■□   5회 □□□□■
        ■ = 평가  ·  □ = 학습

→ 측정값 K개 → 평균 = 안정적인 추정
```

- **측정값 1개 → 측정값 K개**
- 평균 = 안정적인 추정 · 표준편차 = 그 추정의 불확실성
- 모든 데이터가 한 번씩 평가에 사용
- AutoGluon — `num_bag_folds=5`

| 수천 행 이상 | 단순 분할로도 **충분** |
|---|---|
| **수백 행** | 교차검증이 **사실상 필수** |

> ⚠️ **학습 데이터가 늘어나는 것은 아님**
> 300행이면 여전히 240개로 학습 · 그 모델이 K개 만들어질 뿐
> 교차검증이 해결하는 것은 **측정의 문제**

---
## 5. 최종 분할과 점검

- 전처리 결과를 **날짜 경계**에서 시간순으로 분할

In [ ]:
d = clean.sort_values(["Date", "Hour"]).reset_index(drop=True)

days = np.sort(d["Date"].unique())
cut  = days[int(len(days) * 0.8)]

train_data = d[d["Date"] <  cut].reset_index(drop=True)
test_data  = d[d["Date"] >= cut].reset_index(drop=True)

print("기준일:", pd.Timestamp(cut).date())
print("학습용:", train_data.shape)
print("평가용:", test_data.shape)

### 분할 점검 — 무엇을 확인하는가

| 확인 항목 | 기준 |
|---|---|
| 기간이 겹치지 않는가 | 겹치는 날짜 **0** |
| 평가용이 학습용보다 나중인가 | 미래를 예측하는 구조 |
| 평가용 개수가 충분한가 | 수백 개 이상 |

In [ ]:
print("학습:", train_data["Date"].min().date(), "~", train_data["Date"].max().date())
print("평가:", test_data["Date"].min().date(),  "~", test_data["Date"].max().date())
print()
print("겹치는 날짜:", len(set(train_data["Date"]) & set(test_data["Date"])), "일")
print("평가용 개수:", len(test_data), "개")

> ⚠️ **계절 분포는 맞추지 않음**
> 평가용이 가을·겨울에 몰리는 것은 **정상** — 시간순으로 나눴으므로
> 분포를 맞추려 하면 다시 랜덤 분할이 됨

---
## 정리

| 자동 | 사람 |
|---|---|
| 검증용 분할 — 학습용의 10% (`holdout_frac=0.1`) | **평가용 분할** |
| 층화 추출 — 분류 문제 내부 검증 | **묶임의 단위 판단** — 시간 · 개인 · 집단 |
| 교차검증 — `num_bag_folds` | 데이터 크기에 맞는 방법 선택 |

### 결론

- 랜덤 분할 → 성능 부풀림 · 시간순 분할 → 현실적 성능
- 단일 분할 측정값 1개 → 우연인지 실력인지 구분 불가
- **평가용은 최종 보고용** — 보고 되돌아가 고치면 성능이 아니게 됨